# NO POSE CONDITIONING GENERATIONS

In [ ]:
import os
import torch
import gc
from diffusers import StableDiffusionPipeline
from tqdm import tqdm
from pathlib import Path

REPO_ROOT = Path().absolute().parent
DEVICE = "cuda"
UNIQUE_TOKEN = "sks"

MODEL_SAVE_DIR = str(REPO_ROOT / "outputs" / "saved_models" / "baseline")
OUTPUT_BASE_DIR = str(REPO_ROOT / "outputs" / "benchmark-report" / "baseline")
CLASSES_DICT = {
    'humanoid robot': ['unitree']
}
txt_path = REPO_ROOT / "datasets" / "dataset" / "prompts_and_classes.txt"
lines = txt_path.read_text().splitlines()

PROMPT_LIST = []
start_idx = lines.index("Object Prompts")
for line in lines[start_idx:]:
    if line.startswith("'"):
        prompt = line.split("'")[1]
        PROMPT_LIST.append(prompt)
    elif "]" in line:
        break

def load_and_inference(class_name, instance):
    model_path = os.path.join(MODEL_SAVE_DIR, instance)
    output_dir = os.path.join(OUTPUT_BASE_DIR, instance)
    os.makedirs(output_dir, exist_ok=True)

    pipeline = StableDiffusionPipeline.from_pretrained(model_path, torch_dtype=torch.float16, safety_checker=None).to(DEVICE)
    pipeline.set_progress_bar_config(disable=True)

    for i, base_prompt in enumerate(tqdm(PROMPT_LIST)):
        prompt = base_prompt.format(UNIQUE_TOKEN, class_name)
        with torch.no_grad():
            images = pipeline(prompt, num_inference_steps=50, guidance_scale=7.5, num_images_per_prompt=1).images
        
        safe_name = base_prompt.replace(" ", "_").replace("{0}", "").replace("{1}", "")[:30].strip("_")
        for j, img in enumerate(images):
            img.save(os.path.join(output_dir, f"{i:02d}_{safe_name}_{j}.png"))

    del pipeline
    torch.cuda.empty_cache()
    gc.collect()

if __name__ == "__main__":
    for class_name, instances in CLASSES_DICT.items():
        for instance in instances:
            load_and_inference(class_name, instance)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
100%|██████████| 25/25 [01:10<00:00,  2.82s/it]


# T2I-Adapter Generations
## Use for baseline, our_t2iadapter, our_t2iadapter_masked_loss

In [3]:
import os
import torch
from PIL import Image
from diffusers import T2IAdapter, StableDiffusionAdapterPipeline, UniPCMultistepScheduler, UNet2DConditionModel
from transformers import CLIPTextModel
from controlnet_aux import OpenposeDetector
from pathlib import Path

DEVICE = "cuda"
REPO_ROOT = Path().absolute().parent
MODEL_SAVE_DIR = str(REPO_ROOT / "outputs" / "saved_models" / "baseline"/ "unitree")
OUTPUT_DIR = str(REPO_ROOT / "outputs" / "benchmark-report" / "our_t2iadapter")
UNIQUE_TOKEN = "sks"
CLASS_NAME = "humanoid robot"
POSE_PATHS = [
    str(REPO_ROOT / "datasets" / "posing" / "posing-01.jpg"),
    str(REPO_ROOT / "datasets" / "posing" / "posing-04.jpg")
]


txt_path = REPO_ROOT / "datasets" / "dataset" / "prompts_and_classes.txt"
lines = txt_path.read_text().splitlines()

PROMPT_LIST = []
start_idx = lines.index("Object Prompts")
for line in lines[start_idx:]:
    if line.startswith("'"):
        prompt = line.split("'")[1]
        PROMPT_LIST.append(prompt)
    elif "]" in line:
        break

os.makedirs(OUTPUT_DIR, exist_ok=True)

detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
trained_unet = UNet2DConditionModel.from_pretrained(f"{MODEL_SAVE_DIR}/unet", torch_dtype=torch.float16)
trained_text_encoder = CLIPTextModel.from_pretrained(f"{MODEL_SAVE_DIR}/text_encoder", torch_dtype=torch.float16)
adapter = T2IAdapter.from_pretrained("TencentARC/t2iadapter_openpose_sd14v1", torch_dtype=torch.float16)

pipe = StableDiffusionAdapterPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    adapter=adapter,
    unet=trained_unet,
    text_encoder=trained_text_encoder,
    torch_dtype=torch.float16,
    safety_checker=None
).to(DEVICE)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

for pose_idx, pose_path in enumerate(POSE_PATHS, start=1):
    pose_dir = os.path.join(OUTPUT_DIR, f"pose{pose_idx}")
    os.makedirs(pose_dir, exist_ok=True)
    
    original_image = Image.open(pose_path).convert("RGB")
    pose_map = detector(original_image)
    pose_map.save(os.path.join(pose_dir, "input_pose_visualized.png"))
    
    for prompt_idx, base_prompt in enumerate(PROMPT_LIST):
        prompt = base_prompt.format(UNIQUE_TOKEN, CLASS_NAME)
        
        generator = torch.Generator(DEVICE).manual_seed(42)
        
        images = pipe(
            prompt=prompt,
            image=pose_map,
            num_inference_steps=50,
            guidance_scale=7.5,
            adapter_conditioning_scale=0.8,
            num_images_per_prompt=4,
            generator=generator
        ).images
        
        safe_name = base_prompt.replace(" ", "_").replace("{0}", "").replace("{1}", "")[:30].strip("_")
        for img_idx, image in enumerate(images):
            image.save(os.path.join(pose_dir, f"{prompt_idx:02d}_{safe_name}_{img_idx}.png"))

OSError: Unable to load weights from checkpoint file for '/workspace/diffusion-humanoid/ml-interview/outputs/saved_models/baseline/unitree/unet/diffusion_pytorch_model.safetensors' at '/workspace/diffusion-humanoid/ml-interview/outputs/saved_models/baseline/unitree/unet/diffusion_pytorch_model.safetensors'. 

# ControlNet Generations
## Use for baseline, our_controlnet, our_controlnet_masked_loss

In [ ]:
import os
import torch
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline, UniPCMultistepScheduler, UNet2DConditionModel
from transformers import CLIPTextModel
from controlnet_aux import OpenposeDetector
from pathlib import Path

DEVICE = "cuda"
REPO_ROOT = Path().absolute().parent

MODEL_SAVE_DIR = str(REPO_ROOT / "outputs" / "saved_models" / "our_methods_loss_mask_controlnet")
OUTPUT_DIR = str(REPO_ROOT / "outputs" / "benchmark-report" / "our_methods_loss_mask_controlnet_08")

UNIQUE_TOKEN = "sks"
CLASS_NAME = "humanoid robot"

POSE_PATHS = [
    str(REPO_ROOT / "datasets" / "posing" / "posing-01.jpg"),
    str(REPO_ROOT / "datasets" / "posing" / "posing-04.jpg")
]
txt_path = REPO_ROOT / "datasets" / "dataset" / "prompts_and_classes.txt"
lines = txt_path.read_text().splitlines()

PROMPT_LIST = []
start_idx = lines.index("Object Prompts")
for line in lines[start_idx:]:
    if line.startswith("'"):
        prompt = line.split("'")[1]
        PROMPT_LIST.append(prompt)
    elif "]" in line:
        break

os.makedirs(OUTPUT_DIR, exist_ok=True)

detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet")
trained_unet = UNet2DConditionModel.from_pretrained(f"{MODEL_SAVE_DIR}/unet", torch_dtype=torch.float16)
trained_text_encoder = CLIPTextModel.from_pretrained(f"{MODEL_SAVE_DIR}/text_encoder", torch_dtype=torch.float16)
controlnet = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_openpose", torch_dtype=torch.float16)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    unet=trained_unet,
    text_encoder=trained_text_encoder,
    torch_dtype=torch.float16,
    safety_checker=None
).to(DEVICE)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(disable=True)

for pose_idx, pose_path in enumerate(POSE_PATHS, start=1):
    pose_dir = os.path.join(OUTPUT_DIR, f"pose{pose_idx}")
    os.makedirs(pose_dir, exist_ok=True)
    
    original_image = Image.open(pose_path).convert("RGB")
    pose_map = detector(original_image)
    pose_map.save(os.path.join(pose_dir, "input_pose_visualized.png"))
    
    for prompt_idx, base_prompt in enumerate(PROMPT_LIST):
        prompt = base_prompt.format(UNIQUE_TOKEN, CLASS_NAME)
        
        generator = torch.Generator(DEVICE).manual_seed(42)
        
        images = pipe(
            prompt=prompt,
            image=pose_map,
            num_inference_steps=30,
            guidance_scale=7.5,
            controlnet_conditioning_scale=0.8,
            num_images_per_prompt=4,
            generator=generator
        ).images
        
        safe_name = base_prompt.replace(" ", "_").replace("{0}", "").replace("{1}", "")[:30].strip("_")
        for img_idx, image in enumerate(images):
            image.save(os.path.join(pose_dir, f"{prompt_idx:02d}_{safe_name}_{img_idx}.png"))